In [ ]:
import dynamiqs as dq
dq.set_precision('double')
import numpy as np
from scipy.sparse.linalg import eigs
import scipy.sparse as sp
import scipy.sparse.linalg as spla

import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/Code_python")
import Plot_setting

In [ ]:
# --- constants ---
ħ = 1
e = 1
phi_quantum = 1 / 2

w_a = 25.338776456203686
kappa_b = 5 / 10.4*2
E_J = 37*2*np.pi /2
phi_a = 0.11
phi_b = 0.204
epsilon_p = 0.02
g = np.sin(epsilon_p) * E_J * phi_a**2 * phi_b 
kappa_2 = 4*g**2/kappa_b
kappa_1 = 0.005*kappa_2
kappa_phi = 1e-5

In [ ]:
n_a = 40

# --- operators ---
a = dq.destroy(n_a)
Ia = dq.eye(n_a)

# pure dissipative cat stabilization
H = 0 * Ia
jump_op_1 = np.sqrt(kappa_1)*a
jump_op_phi = np.sqrt(kappa_phi)*dq.dag(a)@a

# --- convert to numpy ---
I_np = np.eye(n_a, dtype=np.complex128)
L_np_1 = jump_op_1.to_numpy()
L_np_phi = jump_op_phi.to_numpy()
H_np = H.to_numpy()

LdL_1 = L_np_1.conj().T @ L_np_1
LdL_phi = L_np_phi.conj().T @ L_np_phi



In [ ]:
alpha_sq = 2
jump_op_2 = np.sqrt(kappa_2) * (a @ a - alpha_sq * Ia)

# --- convert to numpy ---
L_np_2 = jump_op_2.to_numpy()
LdL_2 = L_np_2.conj().T @ L_np_2

# --- Liouvillian in vectorized form (column stacking) ---
liouvillian = (
    -1j * (np.kron(I_np, H_np) - np.kron(H_np.T, I_np))
    + np.kron(L_np_2.conj(), L_np_2)
    - 0.5 * np.kron(I_np, LdL_2)
    - 0.5 * np.kron(LdL_2.T, I_np)
    + np.kron(L_np_phi.conj(), L_np_phi)
    - 0.5 * np.kron(I_np, LdL_phi)
    - 0.5 * np.kron(LdL_phi.T, I_np)
    + np.kron(L_np_1.conj(), L_np_1)
    - 0.5 * np.kron(I_np, LdL_1)
    - 0.5 * np.kron(LdL_1.T, I_np)
)



In [ ]:
vals, vecs = np.linalg.eig(liouvillian)
idx = np.argsort(np.real(vals))

In [ ]:
def vec_to_mat(rho_vec, N):
    return rho_vec.reshape((N, N), order='F')

In [ ]:
dq.plot.wigner(dq.asqarray(vec_to_mat(vecs[:,idx[-2]],n_a)))

In [ ]:
vals, vecs = spla.eigs(liouvillian, k=6, sigma=0, which="LM")
idx = np.argsort(np.real(vals))

In [ ]:
dq.plot.wigner(dq.asqarray(vec_to_mat(vecs[:,idx[-2]],n_a)))

In [ ]:
n_a = 20
n_b = 11
N = n_a * n_b

# Operators
a = dq.destroy(n_a)
b = dq.destroy(n_b)
Ia = dq.eye(n_a)
Ib = dq.eye(n_b)

a_tot = dq.tensor(a, Ib)
b_tot = dq.tensor(Ia, b)

# Correct nonlinear term
term = dq.tensor(a @ a, dq.dag(b))
non_linear_op = term + dq.dag(term)
drive_op = b_tot + b_tot.dag()

La = np.sqrt(kappa_1) * a_tot
Lb = np.sqrt(kappa_b) * b_tot
L_phi = np.sqrt(kappa_phi) * dq.dag(a_tot)@a_tot

# Convert to sparse CSC
La = sp.csc_matrix(La.to_numpy().astype(np.complex128))
Lb = sp.csc_matrix(Lb.to_numpy().astype(np.complex128))
L_phi = sp.csc_matrix(L_phi.to_numpy().astype(np.complex128))
I = sp.identity(N, dtype=np.complex128, format="csc")

def dissipator(L):
    LdL = L.getH() @ L
    return (
        sp.kron(L.conjugate(), L, format="csc")
        - 0.5 * sp.kron(I, LdL, format="csc")
        - 0.5 * sp.kron(LdL.T, I, format="csc")
    )
epsilon_d = 2*alpha_sq*g
H = g * non_linear_op + (epsilon_d / 2) * drive_op
H = sp.csc_matrix(H.to_numpy().astype(np.complex128))

Lio = (
-1j * (sp.kron(I, H, format="csc") - sp.kron(H.T, I, format="csc"))
+ dissipator(La)
+ dissipator(Lb)
)

In [ ]:
vals, vecs = spla.eigs(Lio, k=3, sigma=0, which="LM")
idx = np.argsort(np.real(vals))

In [ ]:
dq.plot.wigner(
    dq.ptrace(
        dq.asqarray(
            vec_to_mat(
                vecs[:,idx[-2]],
                N
                ),
            dims=(n_a,n_b)
            ),
        0
    )
)

In [ ]:
alpha_sq = 6/2
alpha = np.sqrt(alpha_sq)
test_state = dq.coherent_dm(20, 1j*alpha) - dq.coherent_dm(20, -1j*alpha)
dq.plot.wigner(test_state)